# Week 1 Day 2 Lab-Introducing UnderCover

In [1]:
import json
import os
import time
from pathlib import Path
from typing import Literal

import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from openai import OpenAI
from openrouter import OpenRouter
from pydantic import BaseModel, Field


## Connect to OpenRouter

In [4]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = 'https://openrouter.ai/api/v1'

if openrouter_api_key:
    print('OpenRouter API key loaded successfully.')
else: 
    print('No OpenRouter API key found.')

openrouter = OpenAI(api_key=openrouter_api_key,
                        base_url=OPENROUTER_BASE_URL,
                        timeout=120,
                        max_retries=5)

OpenRouter API key loaded successfully.


In [5]:
messages = [
    {
        "role":"system",
        "content":"You are a helpful assistant who answers in the style of a pirate"
    },
    {
        "role": "user",
        "content": "Ahoy, matey! Tell me a pirate joke."
    }
]

response = openrouter.chat.completions.create(
            model="openai/gpt-6-luna",
            messages=messages
        )
display(Markdown(response.choices[0].message.content))


Why did the pirate buy a parrot?

Because it was cheep, matey! 🦜

## Part 1: Load and examine the data

In [17]:
from pathlib import Path
print(Path.cwd())

/Users/michaelperrine/Udemy/Donner/undercover/MyWork/UndercoverProject/week1


In [18]:
DATA = Path("../../../data")

def load(name):
    """ Read one of the JSON seed files"""
    return json.loads((DATA / name).read_text(encoding='utf-8'))

def get_sample_data(tiny=False, held_out=False):
    """ Return a list of claims, and the handlers' decisions on them keyed by claim id.
    
    held_out=True gives the twenty-five held-out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set.
    """
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    if tiny:
        chosen = {decision['claim_id'] for decision in decisions if decision['tiny']}
        claims = [claim for claim in claims if claim['claim_id'] in chosen]
        decisions = [decision for decision in decisions if decision['tiny']]
    labels = {decision['claim_id']: decision for decision in decisions}
    return claims, labels

In [19]:
TINY = False

claims, labels = get_sample_data(tiny=TINY)
policies = {policy['policy_id']: policy for policy in load("policies.json")}
policy_wording = (DATA / "policy_wording.md").read_text(encoding='utf-8')

print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")
print(f"The policy wording is {len(policy_wording.split()):,} words")

25 claims, 25 labels, 50 policies
The policy wording is 1,369 words


In [20]:
claims[0]

{'claim_id': 'CLM-1001',
 'policy_id': 'POL-2001',
 'customer_id': 'CUS-3001',
 'claim_type': 'trip_delay',
 'submitted_on': '2026-06-24',
 'incident_date': '2026-06-18',
 'incident_location': 'Rome, Italy',
 'summary': 'Flight home canceled by the airline, rebooked the next morning',
 'story': 'My flight home from Rome on June 18 was canceled by Meridian Air about two hours before departure because of a crew problem. The next seat they could give me was at 6:55 the following morning. I booked a hotel near the airport, had dinner and breakfast there, and took taxis between the airport and the hotel. I am claiming the hotel, the meals and the taxis, and all of the receipts are attached.',
 'items': [{'description': 'Airport hotel, one night',
   'category': 'accommodation',
   'amount': 245.0},
  {'description': 'Dinner and breakfast', 'category': 'meals', 'amount': 95.0},
  {'description': 'Taxis between the airport and the hotel',
   'category': 'transport',
   'amount': 80.0}],
 'amo